1. make the appropriate data, dictionary but with value type of pandas dataframe
2. construct the DCOPF latex tutorial
3. find out how to handle double line in a bus-pair

In [26]:
import pandas as pd
import glob
excel_files = glob.glob("excel_outputs\*.xlsx")
new_headers_file = pd.ExcelFile("case39.xlsx")
df_39 = {}
for sheet_name in new_headers_file.sheet_names:
    if sheet_name != 'gen' and sheet_name != 'gencost':
        df_39[sheet_name] = pd.read_excel(new_headers_file, sheet_name=sheet_name).columns
    else:
        df_39[sheet_name] = pd.read_excel(new_headers_file, sheet_name=sheet_name).columns[:-2]
for file in excel_files:
    excel_file = pd.ExcelFile(file)
    df_excel = {}
    print(file)
    for sheet_name in excel_file.sheet_names:
        if sheet_name != 'baseMVA':
            df_excel[sheet_name] = pd.read_excel(file, sheet_name=sheet_name)
            df_excel[sheet_name].columns = df_39[sheet_name]
        else:
            df_excel[sheet_name] = pd.read_excel(file, sheet_name=sheet_name, index_col=0)
            df_excel[sheet_name].columns = df_39[sheet_name]
    with pd.ExcelWriter(file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        for sheet_name, df in df_excel.items():
            if sheet_name != 'baseMVA':
                df.to_excel(writer, sheet_name=sheet_name, index=False)
            else:
                df.to_excel(writer, sheet_name=sheet_name, index=True)

excel_outputs\pglib_opf_case118_ieee.xlsx
excel_outputs\pglib_opf_case1354_pegase.xlsx
excel_outputs\pglib_opf_case13659_pegase.xlsx
excel_outputs\pglib_opf_case14_ieee.xlsx
excel_outputs\pglib_opf_case162_ieee_dtc.xlsx
excel_outputs\pglib_opf_case1888_rte.xlsx
excel_outputs\pglib_opf_case1951_rte.xlsx
excel_outputs\pglib_opf_case200_pserc.xlsx
excel_outputs\pglib_opf_case2383wp_k.xlsx
excel_outputs\pglib_opf_case240_pserc.xlsx
excel_outputs\pglib_opf_case24_ieee_rts.xlsx
excel_outputs\pglib_opf_case2736sp_k.xlsx
excel_outputs\pglib_opf_case2737sop_k.xlsx
excel_outputs\pglib_opf_case2746wop_k.xlsx
excel_outputs\pglib_opf_case2746wp_k.xlsx
excel_outputs\pglib_opf_case2848_rte.xlsx
excel_outputs\pglib_opf_case2868_rte.xlsx
excel_outputs\pglib_opf_case2869_pegase.xlsx
excel_outputs\pglib_opf_case300_ieee.xlsx
excel_outputs\pglib_opf_case3012wp_k.xlsx
excel_outputs\pglib_opf_case30_as.xlsx
excel_outputs\pglib_opf_case30_fsr.xlsx
excel_outputs\pglib_opf_case30_ieee.xlsx
excel_outputs\pglib_

In [37]:
# from matpowercaseframes import CaseFrames
mpc_file = 'pglib-opf-17.08\\pglib_opf_case39_epri.m'
mpc_frames = CaseFrames(mpc_file)
ppc = {key: mpc_frames.__getattribute__(key) if not isinstance(
    mpc_frames.__getattribute__(key), pd.DataFrame) else mpc_frames.__getattribute__(
    key).values for key in mpc_frames._attributes}
# pandapower.to_excel(net, filename, include_empty_tables=False, include_results=True)
pandapower.to_excel(net, filename, include_empty_tables=False, include_results=True)

In [3]:
import os
import re
import glob
import pandas as pd
import numpy as np

# Directory with .m files
m_dir = 'pglib-opf-17.08'

# Output directory for Excel files
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

# Define header dictionaries for each field
headers = {
    'baseMVA': [0],
    'bus': ['bus_i', 'type', 'Pd', 'Qd', 'Gs', 'Bs', 'area', 'Vm', 'Va', 'baseKV', 'zone', 'Vmax', 'Vmin'],
    # For gen we add two extra columns at the end: EOL and Comments.
    'gen': ['bus_i', 'gen_ID', 'Pg', 'Qg', 'Qmax', 'Qmin', 'Vg', 'mBase', 'status', 
            'Pmax', 'Pmin', 'Pc1', 'Pc2', 'Qc1min', 'Qc1max', 'Qc2min', 'Qc2max', 
            'ramp_agc', 'ramp_10', 'ramp_30', 'ramp_q', 'apf', 'EOL', 'Comments'],
    'branch': ['bus_i', 'bus_j', 'r', 'x', 'b', 'rateA', 'rateB', 'rateC', 'ratio', 'angle', 'status', 'angmin', 'angmax'],
    # For gencost we add two extra columns: EOL and Comments.
    'gencost': ['bus_i', 'gen_ID', 'model', 'startup', 'shutdown', 'n', 'c2', 'c1', 'c0', 'EOL', 'Comments']
}

# Function to parse a matrix from a string block.
def parse_matrix(matrix_str):
    # Remove any comments (%) and blank lines.
    lines = matrix_str.strip().splitlines()
    data = []
    for line in lines:
        # Remove comments (anything after a %)
        line = line.split('%')[0].strip()
        if line:
            # Remove trailing semicolon if present.
            line = line.rstrip(';')
            # Split by whitespace
            row = [float(val) for val in re.split(r'\s+', line) if val]
            data.append(row)
    return data

# Regex patterns for each field
patterns = {
    'baseMVA': r"mpc\.baseMVA\s*=\s*([^;]+);",
    'bus': r"mpc\.bus\s*=\s*\[(.*?)\];",
    'gen': r"mpc\.gen\s*=\s*\[(.*?)\];",
    'gencost': r"mpc\.gencost\s*=\s*\[(.*?)\];",
    'branch': r"mpc\.branch\s*=\s*\[(.*?)\];"
}

# Process each .m file in the folder
m_files = glob.glob(os.path.join(m_dir, "*.m"))

for m_file in m_files:
    with open(m_file, 'r') as f:
        content = f.read()

    data_dict = {}
    
    # Extract baseMVA
    m_baseMVA = re.search(patterns['baseMVA'], content, re.DOTALL)
    if m_baseMVA:
        baseMVA_val = float(m_baseMVA.group(1).strip())
        # Create a one-row DataFrame (or Series) for baseMVA
        df_baseMVA = pd.DataFrame([[baseMVA_val]], columns=headers['baseMVA'])
        data_dict['baseMVA'] = df_baseMVA

    # Extract and parse matrices for the other fields
    for field in ['bus', 'gen', 'gencost', 'branch']:
        match = re.search(patterns[field], content, re.DOTALL)
        if match:
            matrix_str = match.group(1)
            matrix = parse_matrix(matrix_str)
            df = pd.DataFrame(matrix)
            # Adjust DataFrame columns based on expected header length.
            expected_cols = len(headers[field])
            if df.shape[1] < expected_cols:
                # For 'gen' and 'gencost', add extra columns with default values.
                for extra in range(expected_cols - df.shape[1]):
                    df[df.shape[1]] = np.nan
                # Optionally, set default values:
                if field in ['gen', 'gencost']:
                    # EOL column: default to '%'
                    df.iloc[:, -2] = '%'
                    # Comments column: default to empty string.
                    df.iloc[:, -1] = ''
            df.columns = headers[field]
            data_dict[field] = df
        else:
            print(f"Field {field} not found in {m_file}")
    
    # Write to Excel with one sheet per field
    base_name = os.path.splitext(os.path.basename(m_file))[0]
    excel_file = os.path.join(output_dir, base_name + ".xlsx")
    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        for sheet_name, df in data_dict.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"Processed {m_file} -> {excel_file}")

Processed pglib-opf-17.08\pglib_opf_case39_epri.m -> output\pglib_opf_case39_epri.xlsx


In [3]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import math
import pandas as pd

deg_to_rad = math.pi / 180
# Read Excel sheets (adjust sheet names as necessary)
file = "excel_outputs 1/pglib_opf_case14_ieee.xlsx"
bus = pd.read_excel(file, sheet_name='bus').to_numpy()
line = pd.read_excel(file, sheet_name='branch').to_numpy()
gen = pd.read_excel(file, sheet_name='gen').to_numpy()
gen_cost = pd.read_excel(file, sheet_name='gencost').to_numpy()

# Initialize parameters
nEdges = len(line)  # Number of edges
nNodes = len(bus)  # Number of nodes
nGen = len(gen)  # Number of generators
Pbase = 100  # MVA base
Inf_transfer_Pmax = 10e6
Pmax_line = 10e6 / Pbase  # Convert to p.u.
Gmax = max(gen[:, 8]) / Pbase  # p.u.

# Create Gurobi model
model = gp.Model("Economic_Dispatch")

# Decision variables
bus_gen_ss = model.addVars(nNodes, lb=0, ub=Gmax, vtype=GRB.CONTINUOUS, name="bus_gen_ss")
Pij_ss = model.addVars(nEdges, lb=-Pmax_line, ub=Pmax_line, vtype=GRB.CONTINUOUS, name="Pij_ss")
theta_ss = model.addVars(nNodes, lb=-2 * math.pi, ub=2 * math.pi, vtype=GRB.CONTINUOUS, name="theta_ss")

# Power balance constraints
for bus_num_idx in range(nNodes):
    bus_num = bus[bus_num_idx, 0]
    gens = np.where(gen[:, 0] == bus_num)[0].tolist()
    to_bus_list = np.where(line[:, 1] == bus_num)[0].tolist()
    from_bus_list = np.where(line[:, 0] == bus_num)[0].tolist()

    model.addConstr(
        gp.quicksum(bus_gen_ss[np.where(bus[:, 0] == gen[gen_num, 0])[0][0]] for gen_num in gens) +
        gp.quicksum(Pij_ss[to_bus] for to_bus in to_bus_list) -
        gp.quicksum(Pij_ss[from_bus] for from_bus in from_bus_list) ==
        (bus[bus_num_idx, 2] / Pbase) + (bus[bus_num_idx, 4] / Pbase),
        name=f"power_balance_{bus_num}"
    )

# Generator power limit constraints
for gen_num in range(nGen):
    bus_idx = np.where(bus[:, 0] == gen[gen_num, 0])[0][0]
    model.addConstr(bus_gen_ss[bus_idx] <= gen[gen_num, 8] / Pbase, name=f"gen_max_{gen_num}")
    model.addConstr(bus_gen_ss[bus_idx] >= gen[gen_num, 9] / Pbase, name=f"gen_min_{gen_num}")

# Non-generating buses should not generate power
for bus_num_idx in range(nNodes):
    bus_num = bus[bus_num_idx, 0]
    if not np.any(np.equal(gen[:, 0], bus_num)):
        model.addConstr(bus_gen_ss[bus_num_idx] == 0, name=f"no_gen_{bus_num_idx}")

# Active power flow constraints on each line
for line_num in range(nEdges):
    if line[line_num, 8] == 0:
        line[line_num, 8] = 1  # Set tap ratio to 1 for transmission lines

    reciprocal_term = 1 / (line[line_num, 3] * line[line_num, 8])
    from_bus = np.where(bus[:, 0] == line[line_num, 0])[0][0]
    to_bus = np.where(bus[:, 0] == line[line_num, 1])[0][0]

    model.addConstr(
        Pij_ss[line_num] == reciprocal_term * (theta_ss[from_bus] - theta_ss[to_bus] - (line[line_num, 9] * deg_to_rad)),
        name=f"power_flow_{line_num}"
    )

# Thermal limits on lines
for line_num in range(nEdges):
    if line[line_num, 5] == 0:
        line[line_num, 5] = Pmax_line  # Set default line limit

    model.addConstr(Pij_ss[line_num] <= line[line_num, 5] / Pbase, name=f"line_max_{line_num}")
    model.addConstr(Pij_ss[line_num] >= -line[line_num, 5] / Pbase, name=f"line_min_{line_num}")

# Angle difference constraints
for angle_num in range(nEdges):
    from_bus = np.where(bus[:, 0] == line[angle_num, 0])[0][0]
    to_bus = np.where(bus[:, 0] == line[angle_num, 1])[0][0]

    model.addConstr(
        (theta_ss[from_bus] - theta_ss[to_bus]) <= line[angle_num, 12] * deg_to_rad,
        name=f"angle_max_{angle_num}"
    )
    model.addConstr(
        (theta_ss[from_bus] - theta_ss[to_bus]) >= line[angle_num, 11] * deg_to_rad,
        name=f"angle_min_{angle_num}"
    )

# Identifying slack bus and setting its angle to zero
slack_bus = np.where(bus[:, 1] == 3)[0][0]
model.addConstr(theta_ss[slack_bus] == 0, name="slack_bus")

# Objective function: Minimize generation cost
expr = gp.quicksum(
    gen_cost[gen_num, 4] * (bus_gen_ss[np.where(bus[:, 0] == gen[gen_num, 0])[0][0]] * Pbase) +
    gen_cost[gen_num, 5] * (bus_gen_ss[np.where(bus[:, 0] == gen[gen_num, 0])[0][0]] * Pbase) +
    gen_cost[gen_num, 6]
    for gen_num in range(nGen)
)

model.setObjective(expr, GRB.MINIMIZE)
# Disable presolving
model.setParam("Presolve", 0)
# Disable primal heuristics
model.setParam("Heuristics", 0)

# Solve the model
model.optimize()

# Display results
if model.status == GRB.OPTIMAL:
    print("\nOptimal Generation:")
    for k in range(nGen):
        bus_idx = np.where(bus[:, 0] == gen[k, 0])[0][0]
        print(f"G[{gen[k, 0]}] = {bus_gen_ss[bus_idx].X:.6f}")

    print("\nPower Flow:")
    for k in range(nEdges):
        print(f"Pij[{k + 1}] = {Pij_ss[k].X:.6f}")

    print("\nBus Angles:")
    for k in range(nNodes):
        print(f"theta[{k + 1}] = {theta_ss[k].X * 1 / deg_to_rad:.6f} degrees")
else:
    print("Optimization did not reach an optimal solution.")

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
Heuristics  0
Presolve  0

Optimize a model with 134 rows, 48 columns and 245 nonzeros
Model fingerprint: 0xad6173e5
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [3e+02, 3e+02]
  Bounds range     [1e-02, 1e+05]
  RHS range        [1e-02, 7e+00]

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   6.877337e+04   0.000000e+00      0s

Solved in 8 iterations and 0.02 seconds (0.00 work units)
Infeasible model
Optimization did not reach an optimal solution.


In [4]:
# Retrieve basis statuses for the decision variables and constraints
var_basis = model.getAttr(GRB.Attr.VBasis, model.getVars())
constr_basis = model.getAttr(GRB.Attr.CBasis, model.getConstrs())
# Print basis statuses for each variable
print("Variable basis statuses:")
for var, status in zip(model.getVars(), var_basis):
    # Basis status codes typically are:
    # 0: basic, -1: nonbasic at lower bound, -2: nonbasic at upper bound, -3: nonbasic free
    print(f"  {var.VarName}: {status}")
# Print basis statuses for each constraint
print("\nConstraint basis statuses:")
for constr, status in zip(model.getConstrs(), constr_basis):
    print(f"  {constr.ConstrName}: {status}")

Variable basis statuses:
  bus_gen_ss[0]: -1
  bus_gen_ss[1]: -1
  bus_gen_ss[2]: -1
  bus_gen_ss[3]: 0
  bus_gen_ss[4]: 0
  bus_gen_ss[5]: -1
  bus_gen_ss[6]: 0
  bus_gen_ss[7]: -1
  bus_gen_ss[8]: 0
  bus_gen_ss[9]: 0
  bus_gen_ss[10]: 0
  bus_gen_ss[11]: 0
  bus_gen_ss[12]: 0
  bus_gen_ss[13]: 0
  Pij_ss[0]: 0
  Pij_ss[1]: 0
  Pij_ss[2]: 0
  Pij_ss[3]: 0
  Pij_ss[4]: 0
  Pij_ss[5]: 0
  Pij_ss[6]: 0
  Pij_ss[7]: 0
  Pij_ss[8]: 0
  Pij_ss[9]: 0
  Pij_ss[10]: 0
  Pij_ss[11]: 0
  Pij_ss[12]: 0
  Pij_ss[13]: 0
  Pij_ss[14]: 0
  Pij_ss[15]: 0
  Pij_ss[16]: 0
  Pij_ss[17]: 0
  Pij_ss[18]: 0
  Pij_ss[19]: 0
  theta_ss[0]: 0
  theta_ss[1]: 0
  theta_ss[2]: 0
  theta_ss[3]: 0
  theta_ss[4]: 0
  theta_ss[5]: 0
  theta_ss[6]: 0
  theta_ss[7]: -1
  theta_ss[8]: 0
  theta_ss[9]: -1
  theta_ss[10]: -1
  theta_ss[11]: -1
  theta_ss[12]: -1
  theta_ss[13]: -1

Constraint basis statuses:
  power_balance_1.0: -1
  power_balance_2.0: 0
  power_balance_3.0: -1
  power_balance_4.0: 0
  power_balance_5.0: